<a href="https://colab.research.google.com/github/idialloaka-ai/DAILYCHALLENGE/blob/master/Daily_challenge_W6_J2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Défi : Comparaisons de l'attention multiple et du transformateur

Ce notebook implémente les concepts clés de l'architecture Transformer : l'attention par produit scalaire et l'attention multi-têtes.

## 1. Implémentation de l'Attention à tête unique
Objectif : Implémenter le calcul de base $Attention(Q, K, V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class SingleHeadAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        self.query = nn.Linear(d_model, d_model)
        self.key = nn.Linear(d_model, d_model)
        self.value = nn.Linear(d_model, d_model)

    def forward(self, x):
        batch_size, seq_len, d_model = x.shape

        # Projections linéaires
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        # Produit scalaire entrelacé (Scaled Dot-Product)
        # (batch, seq, d) x (batch, d, seq) -> (batch, seq, seq)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_model)

        # Softmax pour obtenir les poids d'attention
        weights = F.softmax(scores, dim=-1)

        # Application des poids aux valeurs
        # (batch, seq, seq) x (batch, seq, d) -> (batch, seq, d)
        output = torch.matmul(weights, V)

        return output, weights

# Validation avec des tenseurs factices
d_model = 128
sha = SingleHeadAttention(d_model)
dummy_input = torch.randn(8, 20, d_model) # (batch, seq_len, d_model)
output, weights = sha(dummy_input)
print(f"Forme de sortie: {output.shape}")
print(f"Forme des poids d'attention: {weights.shape}")

## 2. Module d'Attention Multi-Têtes (Multi-Head Attention)
Ici, nous divisons l'espace de dimension en plusieurs têtes pour permettre au modèle de se concentrer sur différentes parties de la séquence simultanément.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model doit être divisible par num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.query = nn.Linear(d_model, d_model)
        self.key = nn.Linear(d_model, d_model)
        self.value = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, seq_len, d_model = x.shape

        # 1. Projections et division en têtes
        # (B, S, D) -> (B, S, H, D_K) -> (B, H, S, D_K)
        q = self.query(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        k = self.key(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        v = self.value(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

        # 2. Attention par produit scalaire entrelacé
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # 3. Concaténation
        # (B, H, S, D_K) -> (B, S, H, D_K) -> (B, S, D)
        attn_output = torch.matmul(attn_weights, v)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)

        # 4. Projection finale de sortie
        return self.out_proj(attn_output)

mha = MultiHeadAttention(d_model=128, num_heads=8)
output = mha(dummy_input)
print(f"Sortie MHA shape: {output.shape}")

## 3. Pile d'encodeur et Entraînement (Optionnel)
Structure simplifiée d'un bloc Transformer pour la classification NLI.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, d_model)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Connexion résiduelle + Norm
        attn_out = self.attention(x)
        x = self.norm1(x + self.dropout(attn_out))

        # Feed Forward + Norm
        ff_out = self.ff(x)
        x = self.norm2(x + self.dropout(ff_out))
        return x

# Exemple d'architecture pour NLI (3 classes : entailment, neutral, contradiction)
class MiniTransformerNLI(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, ff_dim, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([TransformerBlock(d_model, num_heads, ff_dim) for _ in range(num_layers)])
        self.classifier = nn.Linear(d_model, 3)

    def forward(self, x):
        x = self.embedding(x)
        for block in self.blocks:
            x = block(x)
        # Pooling simple : prendre la moyenne des tokens
        x = x.mean(dim=1)
        return self.classifier(x)

# Initialisation
model = MiniTransformerNLI(vocab_size=1000, d_model=128, num_heads=4, ff_dim=256, num_layers=2)
print(model)